In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

print("Environment ready!")

Environment ready!


In [2]:
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")

In [3]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [4]:
orders.shape

(99441, 8)

In [5]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [6]:
#Convert dates from string to datetime

date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

print("Date conversion completed!")

Date conversion completed!


In [7]:
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [8]:
orders[date_columns].isna().sum()

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [9]:
#missing dates associated with particular order statuses

orders.groupby('order_status').agg(
    total_orders=('order_id', 'count'),
    missing_approval=('order_approved_at', lambda x: x.isna().sum()),
    missing_carrier=('order_delivered_carrier_date', lambda x: x.isna().sum()),
    missing_delivery=('order_delivered_customer_date', lambda x: x.isna().sum())
).sort_values('total_orders', ascending=False)

,total_orders,missing_approval,missing_carrier,missing_delivery
order_status,,,,
delivered,96478,14,2,8
shipped,1107,0,0,1107
canceled,625,141,550,619
unavailable,609,0,609,609
invoiced,314,0,314,314
processing,301,0,301,301
created,5,5,5,5
approved,2,0,2,2


In [10]:
#check duplicate order

orders['order_id'].duplicated().sum()

np.int64(0)

In [11]:
#check duplicate customers

orders['customer_id'].nunique()

99441

In [12]:
#how many orders does each customer have?\

orders.groupby('customer_id').size().value_counts().sort_index()

1    99441
Name: count, dtype: int64

In [13]:
# Load customer data to identify unique customers across multiple order

customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [14]:
# Count unique customers to determine the true customer base for churn analysis

customers['customer_unique_id'].nunique()

96096

In [15]:
#Identify customers with more than one order to measure repeat purchasing behavior

customer_order_counts = orders.merge(
    customers[['customer_id', 'customer_unique_id']],
    on='customer_id',
    how='left'
).groupby('customer_unique_id').size()

# Calculate the percentage of unique customers who placed more than one order
repeat_customer_rate = (customer_order_counts > 1).mean() * 100

repeat_customer_rate

np.float64(3.1187562437562435)

In [16]:
#Count customers with one order versus customers with multiple orders


customer_order_counts.value_counts().sort_index()

1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [17]:
#Convert customer order counts into a customer-level table for further analysis

customer_summary = customer_order_counts.reset_index(name='order_count')

customer_summary.head()

,customer_unique_id,order_count
0,0000366f3b9a7992bf8c76cfdf3221e2,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1
2,0000f46a3911fa3c0805444483337064,1
3,0000f6ccb0745a6a4b88665a16c9f078,1
4,0004aac84e0df4da2b147fca70cf8255,1


In [18]:
#Calculate each customer's first and most recent purchase date for retention and churn analysis

purchase_dates = orders.merge(
    customers[['customer_id', 'customer_unique_id']],
    on='customer_id',
    how='left'
).groupby('customer_unique_id')['order_purchase_timestamp'].agg(
    first_purchase='min',
    last_purchase='max'
)

purchase_dates.head()

,first_purchase,last_purchase
customer_unique_id,,
0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,2018-05-10 10:56:27
0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,2018-05-07 11:11:27
0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,2017-03-10 21:05:03
0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,2017-10-12 20:29:41
0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,2017-11-14 19:45:42


In [19]:
#Combine order frequency and purchase dates into one customer-level analysis table

customer_summary = customer_summary.merge(
    purchase_dates,
    on='customer_unique_id',
    how='left'
)

customer_summary.head()

,customer_unique_id,order_count,first_purchase,last_purchase
0,0000366f3b9a7992bf8c76cfdf3221e2,1,2018-05-10 10:56:27,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,2018-05-07 11:11:27,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,1,2017-03-10 21:05:03,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,1,2017-10-12 20:29:41,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,1,2017-11-14 19:45:42,2017-11-14 19:45:42


In [20]:
#Calculate the number of days between each customer's first and most recent purchase

customer_summary['customer_lifetime_days'] = (
    customer_summary['last_purchase'] - customer_summary['first_purchase']
).dt.days

customer_summary.head()

,customer_unique_id,order_count,first_purchase,last_purchase,customer_lifetime_days
0,0000366f3b9a7992bf8c76cfdf3221e2,1,2018-05-10 10:56:27,2018-05-10 10:56:27,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,2018-05-07 11:11:27,2018-05-07 11:11:27,0
2,0000f46a3911fa3c0805444483337064,1,2017-03-10 21:05:03,2017-03-10 21:05:03,0
3,0000f6ccb0745a6a4b88665a16c9f078,1,2017-10-12 20:29:41,2017-10-12 20:29:41,0
4,0004aac84e0df4da2b147fca70cf8255,1,2017-11-14 19:45:42,2017-11-14 19:45:42,0


In [21]:
#Identify the earliest and latest order dates to understand the analysis period

orders['order_purchase_timestamp'].agg(['min', 'max'])

min   2016-09-04 21:15:19
max   2018-10-17 17:30:18
Name: order_purchase_timestamp, dtype: datetime64[ns]

In [22]:
#Set the reference date one day after the last recorded order for customer recency analysis

reference_date = orders['order_purchase_timestamp'].max().normalize() + pd.Timedelta(days=1)

reference_date

Timestamp('2018-10-18 00:00:00')

In [23]:
#Calculate how many days each customer has been inactive since their last purchase

customer_summary['recency_days'] = (
    reference_date - customer_summary['last_purchase'].dt.normalize()
).dt.days

#Recency = number of days since the customer's most recent purchase.

#Preview customer recency values before using them for segmentation
customer_summary[['customer_unique_id', 'last_purchase', 'recency_days']].head()

,customer_unique_id,last_purchase,recency_days
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,161
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,164
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,587
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,371
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,338


In [24]:
#Rename order count to frequency to represent how often each customer purchased
customer_summary['frequency'] = customer_summary['order_count']

#Frequency = How many times did a customer purchase during the period we're analyzing?

customer_summary[['customer_unique_id', 'frequency']].head()

,customer_unique_id,frequency
0,0000366f3b9a7992bf8c76cfdf3221e2,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1
2,0000f46a3911fa3c0805444483337064,1
3,0000f6ccb0745a6a4b88665a16c9f078,1
4,0004aac84e0df4da2b147fca70cf8255,1


In [25]:
#Load payment data to calculate the total monetary value generated by each customer
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [26]:
#Check how many payment records each order has before calculating total customer spending
payments.groupby('order_id').size().value_counts().sort_index()

1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64

In [27]:
#Sum all payment records for each order to get the complete amount paid for that order
order_payment_totals = payments.groupby('order_id')['payment_value'].sum()

order_payment_totals.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214     72.19
00018f77f2f0320c557190d7a144bdd3    259.83
000229ec398224ef6ca0657da4fc703e    216.87
00024acbcdf0a6daa1e931b038114c75     25.78
00042b26cf59d7ce69dfabb4e55b4fd9    218.04
Name: payment_value, dtype: float64

In [28]:
#Connect each order's total payment to the unique customer who placed the order
order_customer_payments = orders[
    ['order_id', 'customer_id']
].merge(
    customers[['customer_id', 'customer_unique_id']],
    on='customer_id',
    how='left'
).merge(
    order_payment_totals.rename('order_payment'),
    on='order_id',
    how='left'
)

order_customer_payments.head()

,order_id,customer_id,customer_unique_id,order_payment
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,28.62


In [29]:
#Sum all order payments for each unique customer to calculate total customer spending
customer_monetary = (
    order_customer_payments
    .groupby('customer_unique_id')['order_payment']
    .sum()
    .rename('monetary')
)

customer_monetary.head()

customer_unique_id
0000366f3b9a7992bf8c76cfdf3221e2    141.90
0000b849f77a49e4a4ce2b2a4ca5be3f     27.19
0000f46a3911fa3c0805444483337064     86.22
0000f6ccb0745a6a4b88665a16c9f078     43.62
0004aac84e0df4da2b147fca70cf8255    196.89
Name: monetary, dtype: float64

In [36]:
#Combine monetary value with recency and frequency to create the complete RFM customer table
customer_summary = customer_summary.merge(
    customer_monetary,
    on='customer_unique_id',
    how='left'
)

# Verify that all three RFM metrics are present for each customer
customer_summary[
    ['customer_unique_id', 'recency_days', 'frequency', 'monetary']
].head()

,customer_unique_id,recency_days,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,161,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,164,1,27.19
2,0000f46a3911fa3c0805444483337064,587,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,371,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,338,1,196.89


In [37]:
#Rank customers into five Recency groups so we can compare customer activity levels
customer_summary['R_score'] = pd.qcut(
    customer_summary['recency_days'],
    q=5,
    labels=[5, 4, 3, 2, 1]
)

#Check how customers are distributed across the five Recency scores
customer_summary['R_score'].value_counts().sort_index()

R_score
5    19285
4    19261
3    19231
2    19137
1    19182
Name: count, dtype: int64

In [38]:
#Rank customers into five Frequency groups based on how often they purchased
customer_summary['F_score'] = pd.qcut(
    customer_summary['frequency'].rank(method='first'),
    q=5,
    labels=[1, 2, 3, 4, 5]
)

# Check how customers are distributed across the five Frequency scores
customer_summary['F_score'].value_counts().sort_index()

F_score
1    19220
2    19219
3    19219
4    19219
5    19219
Name: count, dtype: int64

In [39]:
#Rank customers into five Monetary groups based on how much they spent
customer_summary['M_score'] = pd.qcut(
    customer_summary['monetary'].rank(method='first'),
    q=5,
    labels=[1, 2, 3, 4, 5]
)

#Check how customers are distributed across the five Monetary scores
customer_summary['M_score'].value_counts().sort_index()

M_score
1    19220
2    19219
3    19219
4    19219
5    19219
Name: count, dtype: int64

In [40]:
#Combine Recency, Frequency, and Monetary scores into one overall RFM score
customer_summary['RFM_score'] = (
    customer_summary['R_score'].astype(int).astype(str) +
    customer_summary['F_score'].astype(int).astype(str) +
    customer_summary['M_score'].astype(int).astype(str)
)

#Preview the combined RFM scores for validation
customer_summary[
    ['customer_unique_id', 'R_score', 'F_score', 'M_score', 'RFM_score']
].head()

,customer_unique_id,R_score,F_score,M_score,RFM_score
0,0000366f3b9a7992bf8c76cfdf3221e2,4,1,4,414
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4,1,1,411
2,0000f46a3911fa3c0805444483337064,1,1,2,112
3,0000f6ccb0745a6a4b88665a16c9f078,2,1,1,211
4,0004aac84e0df4da2b147fca70cf8255,2,1,4,214


In [41]:
#Convert RFM scores into business-friendly customer segments for retention analysis
def assign_segment(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4 and row['M_score'] >= 4:
        return 'Champions'
    elif row['R_score'] >= 3 and row['F_score'] >= 3:
        return 'Loyal Customers'
    elif row['R_score'] <= 2 and row['F_score'] >= 4 and row['M_score'] >= 4:
        return "Can't Lose Them"
    elif row['R_score'] <= 2 and row['F_score'] >= 3:
        return 'At Risk'
    elif row['R_score'] <= 2 and row['F_score'] <= 2:
        return 'Hibernating'
    else:
        return 'Potential Loyalists'

#Apply the segmentation rules to every customer
customer_summary['segment'] = customer_summary.apply(assign_segment, axis=1)

#Count customers in each segment to understand the customer portfolio
customer_summary['segment'].value_counts()

segment
Loyal Customers        28065
Potential Loyalists    23013
At Risk                16766
Hibernating            15426
Champions               6699
Can't Lose Them         6127
Name: count, dtype: int64

In [42]:
# Export customer RFM segmentation results
customer_summary.to_csv(
    "rfm_customer_segments.csv",
    index=False
)

In [43]:
#Measure customer count and revenue contribution for each RFM segment
segment_summary = customer_summary.groupby('segment').agg(
    customers=('customer_unique_id', 'count'),
    total_revenue=('monetary', 'sum'),
    avg_revenue_per_customer=('monetary', 'mean')
).sort_values('total_revenue', ascending=False)

#Preview the business performance of each customer segment
segment_summary

,customers,total_revenue,avg_revenue_per_customer
segment,,,
Loyal Customers,28065,3770040.08,134.332445
Potential Loyalists,23013,3713519.12,161.366146
Hibernating,15426,2532878.43,164.195412
Champions,6699,2127561.60,317.593910
Can't Lose Them,6127,1942737.25,317.078056
At Risk,16766,1922135.64,114.644855


In [44]:
#Calculate each segment's percentage contribution to total customer revenue
segment_summary['revenue_share_pct'] = (
    segment_summary['total_revenue']
    / segment_summary['total_revenue'].sum()
) * 100

#Sort segments by their revenue contribution to identify the most valuable groups
segment_summary.sort_values('revenue_share_pct', ascending=False)

,customers,total_revenue,avg_revenue_per_customer,revenue_share_pct
segment,,,,
Loyal Customers,28065,3770040.08,134.332445,23.549692
Potential Loyalists,23013,3713519.12,161.366146,23.196632
Hibernating,15426,2532878.43,164.195412,15.821717
Champions,6699,2127561.60,317.593910,13.289891
Can't Lose Them,6127,1942737.25,317.078056,12.135379
At Risk,16766,1922135.64,114.644855,12.006690


In [45]:
#Understand how long customers typically go between purchases so we can choose a defensible churn threshold
repeat_customers = customer_summary[
    customer_summary['frequency'] > 1
].copy()

#Calculate the average number of days between purchases for repeat customers
repeat_customers['avg_days_between_orders'] = (
    repeat_customers['customer_lifetime_days']
    / (repeat_customers['frequency'] - 1)
)

#Summarize the distribution of customer purchase intervals
repeat_customers['avg_days_between_orders'].describe()

count    2997.000000
mean       79.731839
std       108.014416
min         0.000000
25%         0.000000
50%        31.000000
75%       122.000000
max       608.000000
Name: avg_days_between_orders, dtype: float64

In [46]:
#Classify customers as churned when they have been inactive for 90 days or more
customer_summary['churned'] = (
    customer_summary['recency_days'] >= 90
)

#Calculate the percentage of customers classified as churned
churn_rate = customer_summary['churned'].mean() * 100

#Display the overall churn rate
churn_rate

np.float64(90.55423742923743)

In [47]:
#Calculate the total historical revenue generated by customers classified as churned
churned_revenue = customer_summary.loc[
    customer_summary['churned'],
    'monetary'
].sum()

#Calculate the total historical revenue generated by all customers
total_customer_revenue = customer_summary['monetary'].sum()

#Measure what percentage of historical customer revenue comes from churned customers
churned_revenue_share = (
    churned_revenue / total_customer_revenue
) * 100

#Display the key churn revenue metrics
churned_revenue, total_customer_revenue, churned_revenue_share

(np.float64(14496601.07),
 np.float64(16008872.12),
 np.float64(90.55354406816262))

In [48]:
#Estimate potential revenue exposure under different customer retention scenarios
revenue_at_risk = pd.DataFrame({
    'scenario': ['10%', '20%', '30%'],
    'revenue_at_risk': [
        churned_revenue * 0.10,
        churned_revenue * 0.20,
        churned_revenue * 0.30
    ]
})

#Display the estimated revenue exposure under each scenario
revenue_at_risk

,scenario,revenue_at_risk
0,10%,1449660.107
1,20%,2899320.214
2,30%,4348980.321


In [49]:
# Export customer RFM segmentation results
customer_summary.to_csv(
    "rfm_customer_segments.csv",
    index=False
)

In [50]:
import os

os.path.abspath("rfm_customer_segments.csv")

'C:\\Users\\navee\\Desktop\\project-1\\ecommerce-customer-analytics\\notebook\\rfm_customer_segments.csv'